# Task 3 — Thực nghiệm bộ giải Kakuro bằng CSP

## 1. Mô hình hóa thành CSP

Một CSP là bộ ba **(X, D, C)**:

- **Variables (X):** mỗi ô trắng là một biến `X_(r,c)` theo tọa độ bắt đầu từ 0. Một ô thuộc đồng thời một run ngang và một run dọc; hai run giao nhau tại cùng biến.
- **Domains (D):** ban đầu `D_(r,c) = {1,2,...,9}` (ô cho sẵn, nếu có, sẽ có miền singleton).
- **Constraints (C):** với mỗi run `R=(X1,...,Xk)` và clue `T`, `X1 + ... + Xk = T` đồng thời `AllDifferent(X1,...,Xk)`. Đây là ràng buộc n-ary; cả tổng và không lặp phải đúng trong cùng run.


In [2]:
from pathlib import Path
import sys

from time import perf_counter

project_dir = Path.cwd()
if not (project_dir / "boards.py").is_file():
    project_dir = project_dir / "task3"
if not (project_dir / "boards.py").is_file():
    raise FileNotFoundError("Không tìm thấy task3/boards.py; hãy mở notebook trong repository.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from boards import make_assignment_board, make_demo_board
from kakuro_solver import KakuroSolver

board = make_assignment_board()
model = KakuroSolver(board).csp
print(board.describe())
print(f"Số biến (ô trắng): {len(model.variables)}")
print(f"Số ràng buộc run: {len(model.constraints)}")
print(f"Miền ban đầu mỗi ô: {sorted(next(iter(model.initial_domains().values())))}")
print(f"Tổng số cạnh constraint graph: {sum(map(len, model.constraint_graph.values())) // 2}")

Assignment Kakuro 9x9: 48 variables, 28 run constraints.
Số biến (ô trắng): 48
Số ràng buộc run: 28
Miền ban đầu mỗi ô: [1, 2, 3, 4, 5, 6, 7, 8, 9]
Tổng số cạnh constraint graph: 166


## 2. Thuật toán và heuristic


In [4]:
CONFIGURATIONS = [
    ("GAC + MRV + Degree + LCV", "gac"),
    ("Forward Checking + MRV + Degree + LCV", "forward_checking"),
]
REPEATS = 10
benchmark_results = {}

for label, propagation in CONFIGURATIONS:
    trials = []
    for trial in range(1, REPEATS + 1):
        solver = KakuroSolver(
            board,
            propagation=propagation,
            use_mrv=True,
            use_degree=True,
            use_lcv=True,
        )
        started = perf_counter()
        solution = solver.solve()
        wall_ms = (perf_counter() - started) * 1000
        valid = solution is not None and solver.csp.validate_solution(solution)
        assert valid, f"{label}, lượt {trial}: không tìm được nghiệm hợp lệ"
        stats = solver.stats.as_dict()
        trials.append({"wall_ms": wall_ms, "stats": stats})
        print(
            f"{label} | lượt {trial}/{REPEATS} | hợp lệ={valid} | "
            f"{wall_ms:.3f} ms | nodes={stats['expanded_nodes']} | "
            f"assignments={stats['assignments_tried']} | "
            f"backtracks={stats['backtrack_count']}"
        )
    benchmark_results[label] = trials

print("\nTóm tắt (thời gian wall-clock; trung bình ± độ lệch chuẩn mẫu):")
print(f"{'Cấu hình':42} {'Valid':>5} {'ms mean ± sd':>20} {'Nodes':>8} {'Assign':>8} {'Backtrack':>10} {'Removed':>9} {'Contrad.':>9}")
for label, trials in benchmark_results.items():
    times = [trial["wall_ms"] for trial in trials]
    mean_ms = sum(times) / len(times)
    sd_ms = (sum((value - mean_ms) ** 2 for value in times) / (len(times) - 1)) ** 0.5 if len(times) > 1 else 0.0
    mean_stat = lambda key: sum(trial["stats"][key] for trial in trials) / len(trials)
    print(
        f"{label:42} {'Yes':>5} {mean_ms:9.3f} ± {sd_ms:<8.3f} "
        f"{mean_stat('expanded_nodes'):8.1f} {mean_stat('assignments_tried'):8.1f} "
        f"{mean_stat('backtrack_count'):10.1f} {mean_stat('domain_values_removed'):9.1f} "
        f"{mean_stat('contradictions'):9.1f}"
    )

assert all(len(trials) == REPEATS for trials in benchmark_results.values())
print(f"\nKết luận kiểm tra: cả {len(CONFIGURATIONS) * REPEATS} lần chạy đều trả nghiệm 9×9 hợp lệ.")

GAC + MRV + Degree + LCV | lượt 1/10 | hợp lệ=True | 2273.036 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 2/10 | hợp lệ=True | 2242.472 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 3/10 | hợp lệ=True | 2263.124 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 4/10 | hợp lệ=True | 2292.621 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 5/10 | hợp lệ=True | 3176.491 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 6/10 | hợp lệ=True | 3279.044 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 7/10 | hợp lệ=True | 3306.409 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 8/10 | hợp lệ=True | 3559.514 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | lượt 9/10 | hợp lệ=True | 2994.574 ms | nodes=49 | assignments=48 | backtracks=0
GAC + MRV + Degree + LCV | l

## 3. Kiểm tra sự kiện và giao diện trực quan


In [3]:
events = []
demo_solver = KakuroSolver(
    make_demo_board(),
    propagation="gac",
    event_callback=events.append,
)
demo_solution = demo_solver.solve()
assert demo_solution is not None and demo_solver.csp.validate_solution(demo_solution)
event_counts = {}
for event in events:
    event_counts[event.kind] = event_counts.get(event.kind, 0) + 1
print("Nghiệm bảng demo hợp lệ:", demo_solver.csp.validate_solution(demo_solution))
print("Sự kiện do solver phát ra:", event_counts)
assert "assignment" in event_counts and "solution_found" in event_counts

Nghiệm bảng demo hợp lệ: True
Sự kiện do solver phát ra: {'domain_reduced': 13, 'variable_selected': 4, 'assignment': 4, 'solution_found': 1}
